# Exploration du fichier de janvier 2024

**Objectif.** Comprendre le contenu et la qualité de `yellow_tripdata_2024-01.parquet`
(courses des taxis jaunes de New York, source : NYC TLC) avant de concevoir les règles
de la couche Silver.

**Méthode.** Pour chaque section : une question, une mesure, un constat.
Les décisions qui en découlent sont consignées dans `docs/regles_qualite.md`.

In [2]:
import pandas as pd
import pyarrow.parquet as pq

chemin = "../data/raw/yellow_tripdata_2024-01.parquet"
df = pd.read_parquet(chemin)
total = len(df)

## 1. Schéma

**Question.** Quelles colonnes contient le fichier, et de quels types ?
On lit le schéma stocké dans le fichier Parquet, sans charger les lignes.

In [3]:
fichier = pq.ParquetFile(chemin)
print("Lignes :", fichier.metadata.num_rows)
print("Colonnes :", fichier.metadata.num_columns)
fichier.schema_arrow

Lignes : 2964624
Colonnes : 19


VendorID: int32
tpep_pickup_datetime: timestamp[us]
tpep_dropoff_datetime: timestamp[us]
passenger_count: int64
trip_distance: double
RatecodeID: int64
store_and_fwd_flag: large_string
PULocationID: int32
DOLocationID: int32
payment_type: int64
fare_amount: double
extra: double
mta_tax: double
tip_amount: double
tolls_amount: double
improvement_surcharge: double
total_amount: double
congestion_surcharge: double
Airport_fee: double

**Constat.**
- 2 964 624 lignes et 19 colonnes.
- Les dates sont des `timestamp` sans fuseau horaire.
- Les montants sont des `double` (flottants) : risque d'erreurs d'arrondi, à convertir en `DECIMAL` en Silver.
- `PULocationID` et `DOLocationID` sont des entiers, mais ce sont des **codes de zones**, pas des quantités.
- `Airport_fee` est écrite avec une majuscule, alors que le dictionnaire de données dit `airport_fee` : incohérence de nommage à standardiser.

## 2. Profilage

**Question.** Que contiennent réellement les colonnes ? Valeurs manquantes,
plage de dates, minimum et maximum des distances et des montants.

In [4]:
df.isna().sum()

VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count          140162
trip_distance                 0
RatecodeID               140162
store_and_fwd_flag       140162
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge     140162
Airport_fee              140162
dtype: int64

In [5]:
print(df["tpep_pickup_datetime"].min(), "->", df["tpep_pickup_datetime"].max())

2002-12-31 22:59:39 -> 2024-02-01 00:01:15


In [6]:
df[["trip_distance", "fare_amount", "tip_amount", "total_amount"]].describe()

,trip_distance,fare_amount,tip_amount,total_amount
count,2.964624e+06,2.964624e+06,2.964624e+06,2.964624e+06
mean,3.652169e+00,1.817506e+01,3.335870e+00,2.680150e+01
std,2.254626e+02,1.894955e+01,3.896551e+00,2.338558e+01
min,0.000000e+00,-8.990000e+02,-8.000000e+01,-9.000000e+02
25%,1.000000e+00,8.600000e+00,1.000000e+00,1.538000e+01
50%,1.680000e+00,1.280000e+01,2.700000e+00,2.010000e+01
75%,3.110000e+00,2.050000e+01,4.120000e+00,2.856000e+01
max,3.127223e+05,5.000000e+03,4.280000e+02,5.000000e+03


**Constat.**
- 5 colonnes ont exactement 140 162 valeurs manquantes (4,7 %) : `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge`, `Airport_fee`.
- Les dates de départ vont du 31 décembre 2002 au 1er février 2024 : le fichier contient des dates qui n'appartiennent pas à janvier 2024.
- `trip_distance` va jusqu'à 312 722 miles, ce qui est impossible. La médiane (1,68 mi) est très inférieure à la moyenne (3,65 mi) : les valeurs aberrantes tirent la moyenne vers le haut.
- Des montants sont négatifs (jusqu'à -900 $).

## 3. Valeurs manquantes

**Question.** Les 5 colonnes sont-elles vides sur les mêmes lignes ? Si oui, qu'ont ces lignes en commun ?

In [7]:
cols = ["passenger_count", "RatecodeID", "store_and_fwd_flag",
        "congestion_surcharge", "Airport_fee"]
manquantes = df[cols].isna().all(axis=1)

print("Au moins 1 manque :", df[cols].isna().any(axis=1).sum())
print("Les 5 manquants   :", manquantes.sum())

Au moins 1 manque : 140162
Les 5 manquants   : 140162


In [8]:
print("Courses par fournisseur (tout le fichier) :")
print(df["VendorID"].value_counts().sort_index())
print()
print("Courses par fournisseur (lignes manquantes) :")
print(df[manquantes]["VendorID"].value_counts().sort_index())
print()
print("payment_type des lignes manquantes :")
print(df[manquantes]["payment_type"].value_counts())

Courses par fournisseur (tout le fichier) :
VendorID
1     729732
2    2234632
6        260
Name: count, dtype: int64

Courses par fournisseur (lignes manquantes) :
VendorID
1    48455
2    91447
6      260
Name: count, dtype: int64

payment_type des lignes manquantes :
payment_type
0    140162
Name: count, dtype: int64


In [9]:
print("Lignes avec payment_type = 0 :", (df["payment_type"] == 0).sum())
df["payment_type"].value_counts().sort_index()

Lignes avec payment_type = 0 : 140162


payment_type
0     140162
1    2319046
2     439191
3      19597
4      46628
Name: count, dtype: int64

**Constat.**
- Les 5 colonnes sont vides sur **exactement les mêmes 140 162 lignes**.
- Toutes ces lignes ont `payment_type = 0`, et réciproquement : les 140 162 lignes de type 0 sont celles-là. D'après le dictionnaire, 0 = course **Flex Fare**.
- Les manques ne sont donc pas des erreurs : ce type de course ne remplit pas ces champs. Le fournisseur n° 6 (Myle Technologies) n'a que ce type de course (260 lignes).
- **Décision :** lignes conservées, `NULL` conservés (R8).

## 4. Dates et durées

**Question.** Quelles courses ont un départ hors de janvier 2024 ? Y a-t-il des durées impossibles ?

In [10]:
debut = pd.Timestamp("2024-01-01")
fin = pd.Timestamp("2024-02-01")
hors_mois = ~df["tpep_pickup_datetime"].between(debut, fin, inclusive="left")

print("Départs hors janvier 2024 :", hors_mois.sum())
print((df[hors_mois][["tpep_pickup_datetime", "tpep_dropoff_datetime", "payment_type"]]
    .sort_values("tpep_pickup_datetime")
    .to_string()))

Départs hors janvier 2024 : 18
        tpep_pickup_datetime tpep_dropoff_datetime  payment_type
53119    2002-12-31 22:59:39   2002-12-31 23:05:41             3
53120    2002-12-31 22:59:39   2002-12-31 23:05:41             3
2558056  2009-01-01 00:24:09   2009-01-01 01:13:00             2
1230169  2009-01-01 23:30:39   2009-01-02 00:01:39             2
10915    2009-01-01 23:58:40   2009-01-02 00:01:40             2
369      2023-12-31 23:39:17   2023-12-31 23:42:00             1
753      2023-12-31 23:41:02   2023-12-31 23:48:03             2
3176     2023-12-31 23:47:28   2023-12-31 23:57:07             1
2985     2023-12-31 23:49:12   2024-01-01 00:04:32             1
8628     2023-12-31 23:54:27   2024-01-01 00:13:12             1
2615     2023-12-31 23:56:45   2024-01-01 00:00:28             1
256      2023-12-31 23:56:46   2024-01-01 00:12:06             1
2210     2023-12-31 23:57:17   2024-01-01 00:01:50             1
4137     2023-12-31 23:58:35   2024-01-01 00:13:06         

In [11]:
duree_s = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds()
duree_h = duree_s / 3600

print("durée < 0 :", (duree_s < 0).sum())
print("durée = 0 :", (duree_s == 0).sum())

durée < 0 : 56
durée = 0 : 814


**Constat.**
- 18 lignes ont un départ hors janvier 2024, en quatre groupes : décembre 2002 (2), janvier 2009 (3), 31 décembre 2023 (10), 1er février 2024 (3).
- 2002 et 2009 sont des dates impossibles (horloge de taximètre erronée). Les courses de fin décembre et de début février sont de vraies courses de frontière, qui appartiennent aux mois voisins.
- 56 courses ont une durée négative, et 814 ont une durée nulle.
- **Point d'attention :** les courses de frontière risquent d'être comptées deux fois quand on chargera les autres mois.

## 5. Distances et vitesses

**Question.** Quelles distances sont douteuses ? Une distance de 0 est-elle une erreur ?
On calcule la vitesse implicite (distance / durée) pour juger les valeurs extrêmes,
plutôt que d'utiliser un seuil de distance arbitraire.

In [12]:
dist0 = df["trip_distance"] == 0
meme_zone = df["PULocationID"] == df["DOLocationID"]

print("distance = 0                :", dist0.sum())
print("  dont même zone            :", (dist0 & meme_zone).sum())
print("  dont zones différentes    :", (dist0 & ~meme_zone).sum())
print()
print("payment_type des distances à 0 :")
print(df[dist0]["payment_type"].value_counts().sort_index())

distance = 0                : 60371
  dont même zone            : 25890
  dont zones différentes    : 34481

payment_type des distances à 0 :
payment_type
0    22825
1    20604
2     8583
3     4566
4     3793
Name: count, dtype: int64


In [13]:
mask = (duree_h > 0) & (df["trip_distance"] > 0)
vitesse = df.loc[mask, "trip_distance"] / duree_h[mask]

print(vitesse.describe(percentiles=[.5, .99, .999]))
print()
for seuil in [60, 80, 100, 200]:
    print(f"vitesse > {seuil} mph : {(vitesse > seuil).sum():,}")

count    2.904141e+06
mean     1.405082e+01
std      1.098335e+03
min      4.168934e-04
50%      9.670711e+00
99%      3.712416e+01
99.9%    4.912549e+01
max      1.443334e+06
dtype: float64

vitesse > 60 mph : 1,289
vitesse > 80 mph : 1,107
vitesse > 100 mph : 1,024
vitesse > 200 mph : 830


**Constat.**
- 60 371 courses (2 %) ont une distance de 0. 25 890 partent et arrivent dans la même zone (plausible : course annulée ou très courte). Environ 34 500 ont des zones différentes avec 0 mile, ce qui est incohérent.
- Vitesse implicite : médiane 9,7 mph, centile 99 à 37 mph, centile 99,9 à 49 mph.
- 1 024 courses dépassent 100 mph, dont 830 dépassent 200 mph : valeurs de distance corrompues.
- 265 courses sont entre 60 et 100 mph : zone grise.
- Choisir un seuil entre 60 et 100 mph ne déplace que 265 lignes sur 2,9 millions : le choix est peu sensible.

## 6. Montants

**Question.** Les montants négatifs ou nuls sont-ils des erreurs ?
On les croise avec `payment_type` (0 = Flex Fare, 1 = carte, 2 = espèces, 3 = aucun frais, 4 = contestation).

In [14]:
total_neg = df["total_amount"] < 0
fare_neg = df["fare_amount"] < 0

print("total_amount < 0 :", total_neg.sum())
print(df[total_neg]["payment_type"].value_counts().sort_index())
print()
print("fare_amount < 0  :", fare_neg.sum())
print("  dont total_amount >= 0 :", (fare_neg & ~total_neg).sum())

total_amount < 0 : 35504
payment_type
0        2
1       29
2     8326
3     5741
4    21406
Name: count, dtype: int64

fare_amount < 0  : 37448
  dont total_amount >= 0 : 2064


In [15]:
total_zero = df["total_amount"] == 0

print("total_amount = 0 :", total_zero.sum())
print(df[total_zero]["payment_type"].value_counts().sort_index())

total_amount = 0 : 416
payment_type
0     11
1    129
2     65
3     83
4    128
Name: count, dtype: int64


**Constat.**
- 35 504 courses ont un `total_amount` négatif : 21 406 sont des contestations (type 4) et 5 741 sont sans frais (type 3). Ce sont probablement des ajustements ou des remboursements, pas des erreurs de saisie.
- 8 326 sont payées en espèces et 31 par carte ou Flex Fare : cas à signaler.
- 416 courses ont un total de 0 : seulement 83 sont de type 3. 129 sont payées par carte, ce qui est suspect.
- 37 448 courses ont un `fare_amount` négatif. Parmi elles, 2 064 ont un `total_amount` non négatif : cet écart reste à analyser.

## 7. Conclusions

Le nombre de lignes à rejeter (règles R1, R3 et R4) est calculé dans la cellule ci-dessous.
Plutôt que de supprimer les lignes douteuses (distances à 0, montants négatifs), on distingue
trois statuts : **accepté**, **signalé** et **rejeté**.
Les règles précises sont dans `docs/regles_qualite.md`.

In [16]:
annee = df["tpep_pickup_datetime"].dt.year
r1 = ~annee.isin([2023, 2024])                      # date aberrante
r3 = (duree_s < 0) | ((duree_s == 0) & (df["trip_distance"] > 0))   # durée invalide
vitesse_all = (df["trip_distance"] / duree_h).where(duree_h > 0)
r4 = vitesse_all > 100                              # vitesse impossible

rejet = r1 | r3 | r4
print("R1 date aberrante   :", r1.sum())
print("R3 durée invalide   :", r3.sum())
print("R4 vitesse > 100 mph:", r4.sum())
print(f"Total rejeté (sans doublon entre règles) : {rejet.sum():,} ({rejet.sum() / total:.3%})")

R1 date aberrante   : 5
R3 durée invalide   : 112
R4 vitesse > 100 mph: 1024
Total rejeté (sans doublon entre règles) : 1,141 (0.038%)


**Limites.** Ces constats portent sur un seul mois. Ils seront revalidés avec février et mars.